In [ ]:
# https://www.kaggle.com/datasets/vanshtyagi2o1/dataset-diseases-and-symptoms-trainingtesting?select=Training.csv
import tkinter as tk
import customtkinter as ctk
import pandas as pd
import numpy as np
import spacy
from thefuzz import process
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import threading

In [7]:
# Load NLP - do this globally for performance
try:
    nlp = spacy.load("en_core_web_md")
except:
    import os
    os.system("python -m spacy download en_core_web_md")
    nlp = spacy.load("en_core_web_md")

In [3]:
class MedicalEngine:
    def __init__(self, csv_path):
            # Load and immediately drop empty/phantom columns
            self.df = pd.read_csv(csv_path)
            self.df = self.df.loc[:, ~self.df.columns.str.contains('^Unnamed')]
            
            self.le = LabelEncoder()
            self.df['prognosis'] = self.le.fit_transform(self.df['prognosis'])
            
            self.X = self.df.drop('prognosis', axis=1)
            self.y = self.df['prognosis']
            
            self.model = RandomForestClassifier(n_estimators=250, random_state=42)
            self.model.fit(self.X, self.y)
            
            self.symptoms_list = list(self.X.columns)
            self.clean_symptoms = [s.replace('_', ' ') for s in self.symptoms_list]

    def extract_symptoms(self, text):
        text = text.lower().strip()
        found = []
        for key, value in MEDICAL_MAP.items():
            if key in text:
                found.append(value)
        if not found:
            match, score = process.extractOne(text, self.clean_symptoms)
            if score > 92:
                found.append(self.symptoms_list[self.clean_symptoms.index(match)])
        return list(set(found))

    def get_recommendations(self, current_symptoms, top_n=3):
        """Calculates feature co-occurrence to recommend likely symptoms."""
        if not current_symptoms: return []
        
        # Filter dataset to rows that have ALL currently reported symptoms
        mask = np.ones(len(self.X), dtype=bool)
        for s in current_symptoms:
            if s in self.symptoms_list:
                mask = mask & (self.X[s] == 1)
        
        subset = self.X[mask]
        
        # If no exact match for all, fall back to rows matching ANY symptom
        if len(subset) == 0:
            mask = np.zeros(len(self.X), dtype=bool)
            for s in current_symptoms:
                if s in self.symptoms_list:
                    mask = mask | (self.X[s] == 1)
            subset = self.X[mask]
            
        if len(subset) == 0: return []

        # Sum occurrences of remaining symptoms and get top N
        counts = subset.sum(axis=0).drop(labels=current_symptoms, errors='ignore')
        top_symptoms = counts.nlargest(top_n).index.tolist()
        
        return [s.replace('_', ' ') for s in top_symptoms if counts[s] > 0]
    
    def run_system_test(self, test_file_path):
        print(f"\n--- INITIATING SYSTEM VALIDATION ({test_file_path}) ---")
        try:
            from sklearn.metrics import accuracy_score, classification_report
            
            test_df = pd.read_csv(test_file_path)
            # Drop phantom columns from testing data too
            test_df = test_df.loc[:, ~test_df.columns.str.contains('^Unnamed')] # <--- FIX HERE
            
            X_test = test_df.drop('prognosis', axis=1)
            y_test_text = test_df['prognosis']
            
            y_test_numeric = self.le.transform(y_test_text)
            y_pred = self.model.predict(X_test)
            
            acc = accuracy_score(y_test_numeric, y_pred)
            print(f"STATUS: Validation Complete. Accuracy: {acc * 100:.2f}%")
            print(classification_report(y_test_numeric, y_pred, target_names=self.le.classes_))
            
        except Exception as e:
            print(f"TEST ERROR: {e}")


    def predict_top_two(self, symptoms):
        """Returns the top 2 predicted diseases and their probabilities."""
        if not symptoms: return []
        vector = np.zeros(len(self.symptoms_list))
        for s in symptoms:
            if s in self.symptoms_list:
                vector[self.symptoms_list.index(s)] = 1
        
        prob = self.model.predict_proba([vector])[0]
        # Get indices of top 2 probabilities
        top2_idx = np.argsort(prob)[-2:][::-1] 
        
        results = []
        for idx in top2_idx:
            disease = self.le.inverse_transform([idx])[0]
            confidence = prob[idx] * 100
            results.append((disease, confidence))
            
        return results

In [4]:
class MedicalUI(ctk.CTk):
    def __init__(self, engine):
        super().__init__()
        self.engine = engine
        self.user_symptoms = []

        self.title("AI Medical Assistant")
        self.geometry("650x850")
        ctk.set_appearance_mode("dark")

        # Layout
        self.grid_rowconfigure(0, weight=1)
        self.grid_columnconfigure(0, weight=1)

        # Chat Window
        self.chat_display = ctk.CTkTextbox(self, state="disabled", font=("Segoe UI", 14), wrap="word")
        self.chat_display.grid(row=0, column=0, padx=20, pady=20, sticky="nsew")

        # Input Area
        self.input_frame = ctk.CTkFrame(self)
        self.input_frame.grid(row=1, column=0, padx=20, pady=(0, 10), sticky="ew")

        self.entry = ctk.CTkEntry(self.input_frame, placeholder_text="Enter symptoms here...", height=45)
        self.entry.pack(side="left", fill="x", expand=True, padx=(0, 10))
        self.entry.bind("<Return>", lambda e: self.send_action())

        self.send_btn = ctk.CTkButton(self.input_frame, text="Send", command=self.send_action, width=90, height=45)
        self.send_btn.pack(side="right")

        # Bottom Action Panel
        self.action_frame = ctk.CTkFrame(self, fg_color="transparent")
        self.action_frame.grid(row=2, column=0, padx=20, pady=(0, 20), sticky="ew")
        
        self.manage_btn = ctk.CTkButton(self.action_frame, text="Manage Symptoms", fg_color="#3a3a3a", hover_color="#555555", command=self.open_symptom_manager)
        self.manage_btn.pack(side="left", expand=True, fill="x", padx=(0, 10))

        self.diag_btn = ctk.CTkButton(self.action_frame, text="GENERATE DIAGNOSIS", fg_color="#1a5e1a", hover_color="#124012", command=self.generate_diagnosis_report)
        self.diag_btn.pack(side="right", expand=True, fill="x")

        self.add_log("System", "Online. Describe your symptoms (e.g., 'I have a headache and sore throat').")

    def add_log(self, sender, text):
        self.chat_display.configure(state="normal")
        self.chat_display.insert("end", f"{sender}: {text}\n\n")
        self.chat_display.configure(state="disabled")
        self.chat_display.see("end")

    def send_action(self):
        msg = self.entry.get()
        if not msg: return
        self.add_log("You", msg)
        self.entry.delete(0, "end")
        threading.Thread(target=self.process_input, args=(msg,)).start()

    def process_input(self, text):
        clean_text = text.lower().strip()

        if any(x in clean_text for x in ["diagnose", "done", "finish"]):
            self.after(0, self.generate_diagnosis_report)
            return

        extracted = self.engine.extract_symptoms(clean_text)
        
        if extracted:
            new_symptoms = [s for s in extracted if s not in self.user_symptoms]
            self.user_symptoms.extend(new_symptoms)
            
            if new_symptoms:
                names = ", ".join([n.replace('_', ' ') for n in new_symptoms])
                res = f"Recorded: {names}."
                
                # Fetch dynamically recommended symptoms
                recs = self.engine.get_recommendations(self.user_symptoms)
                if recs:
                    res += f"\n\nPatients with these symptoms often also report:\n• " + "\n• ".join(recs) + "\n\nAre you experiencing any of these?"
            else:
                res = "I already have that on record. Any other symptoms?"
        else:
            if any(g in clean_text for g in ["hi", "hello", "hey"]):
                res = "Hello! Please describe your symptoms so I can help."
            else:
                res = "I didn't recognize a medical symptom. Could you rephrase?"

        self.after(0, lambda: self.add_log("Assistant", res))

    # --- NEW FEATURE: SYMPTOM MANAGER ---
    def open_symptom_manager(self):
        manager = ctk.CTkToplevel(self)
        manager.title("Active Symptoms")
        manager.geometry("350x400")
        manager.grab_set() # Focus lock
        
        ctk.CTkLabel(manager, text="Current Recorded Symptoms", font=("Segoe UI", 16, "bold")).pack(pady=10)
        
        scroll_frame = ctk.CTkScrollableFrame(manager)
        scroll_frame.pack(fill="both", expand=True, padx=10, pady=10)

        if not self.user_symptoms:
            ctk.CTkLabel(scroll_frame, text="No symptoms recorded yet.").pack(pady=20)
            return

        def remove_symptom(sym, frame):
            self.user_symptoms.remove(sym)
            frame.destroy()
            self.add_log("System", f"Removed '{sym.replace('_', ' ')}' from records.")

        for sym in self.user_symptoms:
            row = ctk.CTkFrame(scroll_frame, fg_color="#2b2b2b")
            row.pack(fill="x", pady=2)
            
            ctk.CTkLabel(row, text=sym.replace('_', ' ').title(), font=("Segoe UI", 13)).pack(side="left", padx=10, pady=8)
            del_btn = ctk.CTkButton(row, text="❌", width=30, fg_color="#801919", hover_color="#5c1111", 
                                    command=lambda s=sym, f=row: remove_symptom(s, f))
            del_btn.pack(side="right", padx=10)

    # --- NEW FEATURE: MODERN DIAGNOSIS GRID ---
    def generate_diagnosis_report(self):
        if not self.user_symptoms:
            self.add_log("Assistant", "No symptoms recorded. Please describe how you feel first.")
            return
            
        predictions = self.engine.predict_top_two(self.user_symptoms)
        self.user_symptoms = [] # Clear session after report
        
        report_win = ctk.CTkToplevel(self)
        report_win.title("Preliminary Assessment Report")
        report_win.geometry("700x450")
        report_win.grab_set()

        ctk.CTkLabel(report_win, text="DIAGNOSIS REPORT", font=("Segoe UI", 24, "bold")).pack(pady=(20, 5))
        ctk.CTkLabel(report_win, text="Based on the symptom data provided.", text_color="gray").pack(pady=(0, 20))

        grid_frame = ctk.CTkFrame(report_win, fg_color="transparent")
        grid_frame.pack(fill="both", expand=True, padx=20, pady=10)
        grid_frame.grid_columnconfigure(0, weight=1)
        grid_frame.grid_columnconfigure(1, weight=1)

        # Card 1: Primary Prediction
        card1 = ctk.CTkFrame(grid_frame, fg_color="#1e2a38", corner_radius=15)
        card1.grid(row=0, column=0, padx=10, sticky="nsew")
        ctk.CTkLabel(card1, text="PRIMARY MATCH", font=("Segoe UI", 12, "bold"), text_color="#5ea8ff").pack(pady=(20, 5))
        ctk.CTkLabel(card1, text=predictions[0][0], font=("Segoe UI", 22, "bold")).pack(pady=10)
        
        # Colored confidence indicator
        conf1_color = "#32a852" if predictions[0][1] > 70 else "#a88e32"
        ctk.CTkLabel(card1, text=f"{predictions[0][1]:.1f}% Confidence", font=("Segoe UI", 18), text_color=conf1_color).pack()

        # Card 2: Secondary Prediction
        card2 = ctk.CTkFrame(grid_frame, fg_color="#2a1e20", corner_radius=15)
        card2.grid(row=0, column=1, padx=10, sticky="nsew")
        ctk.CTkLabel(card2, text="SECONDARY MATCH", font=("Segoe UI", 12, "bold"), text_color="#ff7575").pack(pady=(20, 5))
        ctk.CTkLabel(card2, text=predictions[1][0], font=("Segoe UI", 22, "bold")).pack(pady=10)
        ctk.CTkLabel(card2, text=f"{predictions[1][1]:.1f}% Confidence", font=("Segoe UI", 18), text_color="gray").pack()

        ctk.CTkLabel(report_win, text="DISCLAIMER: This system utilizes machine learning for educational pre-consultation.\nIt does not replace professional medical advice. Please consult a physician.", 
                     font=("Segoe UI", 12), text_color="#888888", justify="center").pack(side="bottom", pady=20)

        self.add_log("Assistant", "I have generated your diagnosis report in a new window. Your symptom session has been cleared for your next query.")

In [5]:
# Update your main block to include the test:
if __name__ == "__main__":
    try:
        engine = MedicalEngine("training.csv")
        
        # Run the test before launching the UI
        engine.run_system_test( "testing.csv")
        
        app = MedicalUI(engine)
        app.mainloop()
    except Exception as e:
        print(f"Error: {e}")


--- INITIATING SYSTEM VALIDATION (testing.csv) ---
STATUS: Validation Complete. Accuracy: 97.62%
                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00         1
                                   AIDS       1.00      1.00      1.00         1
                                   Acne       1.00      1.00      1.00         1
                    Alcoholic hepatitis       1.00      1.00      1.00         1
                                Allergy       1.00      1.00      1.00         1
                              Arthritis       1.00      1.00      1.00         1
                       Bronchial Asthma       1.00      1.00      1.00         1
                   Cervical spondylosis       1.00      1.00      1.00         1
                            Chicken pox       1.00      1.00      1.00         1
                    Chronic cholestasis       1.00      1.00      1.00         1
          